**Table of contents**<a id='toc0_'></a>    
- [Preface](#toc1_1_)    
- [What are all these terminologies?](#toc1_2_)    
  - [CPU, Core, Process, and Thread](#toc1_2_1_)    
  - [GIL in python](#toc1_2_2_)    
  - [Multithreading and Multiprocessing](#toc1_2_3_)    
- [GIL becomes optional starting from python3.13+](#toc1_3_)    
- [Speed with vs without GIL](#toc1_4_)    
- [Summary](#toc1_5_)    
- [References](#toc1_6_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Preface](#toc0_)

When buying computers, we often hear often, e.g., Laptop 16 inch with [i7-13700K Processor](https://www.intel.com/content/www/us/en/products/sku/230500/intel-core-i713700k-processor-30m-cache-up-to-5-40-ghz/specifications.html) (13th Generation), 16 cores. From this, we generally know that one CPU may contain multiple cores. But what about processes and treads? In my daily job I rarely touch these core concepts since most packages alraedy handels well for you. I only know roughly that multitreading and multiprocessing help on speeding up the application in a smart way, but how, I don't know. Nevertheless, when people are discussing that python will gradually relax the GIL (Global Interpreter Lock) which affects the multithreading, it intrigues me a lot and I want to know better the idea behind it.

At the beginning these technology specifications seems to be overwhelming, but actually the core concept is not that difficult to understand as I read more articles. 
There are several online discussions or posts that have sometimes different optinions, which might also cause confusion. This is why **I want to put all I have read and learned into one post (with experimentation to find out the truth)** and hope these will also help you save time if you have same questions as I did.



## <a id='toc1_2_'></a>[What are all these terminologies?](#toc0_)

### <a id='toc1_2_1_'></a>[CPU, Core, Process, and Thread](#toc0_)

I hope with one graph and a legend displayed below, it can clearly explains to you what each of them means.

<pre>
🧠 CPU
 ├── 🧩 Core 1
 │    ├── 🏭 Process A
 │    │    ├── 🧵 Thread A1
 │    │    ├── 🧵 Thread A2
 │    │    └── 🧵 Thread A3
 │    └── 🏭 Process B
 │         ├── 🧵 Thread B1
 ├── 🧩 Core 2
 │    └── 🏭 Process C
 │         ├── 🧵 Thread C1
 │         └── 🧵 Thread C2
 └── ...
</pre>


🧠 CPU → The entire processor. 1 CPU contains one to multiple cores.

🧩 Core → Independent execution unit (can run one or more threads).

🏭 Process → Independent program with its own memory space in execution.
- The operating system schedules processes to run on CPU cores. 
- Each process has itws own memory space and system resources, i.e., memory is not shared across processes. <br>
- And based on the graph, multiple processes can be distributed across multiple cores.

🧵 Thread → A smaller unit of execution inside a process.
- Threads within the same process share memory and resources.
- Threads are lighter than processes and switch faster, but they can interfere with each other if not synchronized properly within the same process.


One vivid example that I would like to quote describes well how process and thread differ:

> **Quote:**  
> Example: Microsoft Word <br>
> When you open Word, you create a process. When you start typing, the process spawns threads: <br>
> one to read keystrokes, another to display text, one to autosave your file, and yet another to highlight spelling mistakes. <br>
> By spawning multiple threads, Microsoft takes advantage of idle CPU time (waiting for keystrokes or files to load) and makes you more productive.
>
> **Source:** [Brendan Fortuner - Intro to Thraeds and Processes in Python](https://medium.com/@bfortuner/python-multithreading-vs-multiprocessing-73072ce5600b)


### <a id='toc1_2_2_'></a>[GIL in python](#toc0_)

Now let us talk about the global interpreter lock (GIL) in python.
So what is GIL, and how does it affect multithreading?

Global Interpreter Lock (GIL) is a mutex (or a piece of code) in CPython. It ensures only one thread executes CPython bytecode at a time. 
Let us have a look at the official documentation regarding its interpretation:

> **Quote:**  
> “The mechanism used by the CPython interpreter to assure that only one thread executes Python bytecode at a time.  
> This simplifies the CPython implementation by making the object model (including critical built-in types such as `dict`) implicitly safe against concurrent access.  
> Locking the entire interpreter makes it easier for the interpreter to be multi-threaded, at the expense of much of the parallelism afforded by multi-processor machines.”
>
> **Source:** [Python Docs – Global Interpreter Lock](https://docs.python.org/3/glossary.html#term-global-interpreter-lock)


When you install Python, it is standard to install CPython that is written in C since CPython helps compile your `.py` code into bytecode and runs it on the CPython interpreter (a virtual machine) and helps manage memory, garbage collection and the GIL.

From the quote we know that GIL:
- defines the behavior other implementations should follow. 
- simplifies memory management (as multi-thread within the same process would share the same memory and you need to be careful that one thread doesn't override another thread's result.)
- but it limits true parallelism.

There are other implementations for comparison, just FYI:
- Jython: runs on the Java Virtual Machine and uses Java's thread model which is fully multithreaded and has no GIL.
- IronPython: runs on .NET CLR, using .NET's thread-safe environment and has no GIL.


### <a id='toc1_2_3_'></a>[Multithreading and Multiprocessing](#toc0_)


🧵🧵🧵 Now we can better understand the mechanism of multithreading:
- it runs multiple threads within the same process, and they share the same memory space.
- due to GIL, only one thread executes Python bytecode at a time. Therefore, multithreading runs threads **NOT in a PARALLEL way, BUT in a CONCURRENT way**! 😱😱😱
  - Concurrent: Code that can be executed out of order but still only one thread gets executed at any given time.
  - Parallel: Capability to execute code at the same time.
- so multithreading doesn't speed up CPU-bound tasks (e.g., heavy math calculation, image/video compression), but **best for I/O-bound tasks** (e.g., read/write files, network requests HTTP APIs, database queries, download/upload files)


🏭🏭🏭 On the ohter hand, for multiprocessing:
- it runs multiple processes, each with its own Python interpreter and memory space.
- Not limited by GIL -- each process can run on a separate CPU core.
- In contrast to multithreading, multiprocessing is best for CPU-bound tasks (and also not bad for I/O-bound tasks).



References:
- The article [Difference Between Concurrency and Parallelism by TechDifferences](https://techdifferences.com/difference-between-concurrency-and-parallelism.html) provides a good overview (also with a YouTube video within) of the differences between concurrency and parallelism. I have summarized the most important termins in one sentence interpretation in the above, but you can refer to this articel for more vivid details.
 
- The article [Multithreading VS Multiprocessing in Python by Amine Baatout ](https://medium.com/contentsquare-engineering-blog/multithreading-vs-multiprocessing-in-python-ece023ad55a) is highly recommended to be read through:
  
  - it deomnstrates that multithreading doesn't run in parallel but concurrently via the [experiment code](https://github.com/wkCircle/multithreading-vs-multiprocessing/blob/master/live_comparison.py), which I have also forked and run in my machine (but needs `if __name__ == "__main__"` to avoid BrokenProcessPool error when compared to the original code), and plotted it below as Fig.1. <br>
  From Fig.1, **we can confirm that multithreading indeed runs threads neither in parallel nor in sequential. They are run concurrently: at any given time, only one thread proceeds a little, and the other takes on.**
  <style>
  figcaption {
    color: white;
    font-style: italic;
    padding: 2px;
    text-align: center;
  }
  </style>
  <figure>
    <img src="./img/gil-lock-multithreading-vs-multiprocessing-fig1.png" alt="multithreading runs concurrenctly while mutlprocessing runs parallely">
    <figcaption>Fig.1: multithreading runs concurrenctly while mutlprocessing runs parallely</figcaption>
  </figure>

  - it demonstrates that multithreading performs badly on CPU-heavy tasks (can be even worse than single thread sequential execution). You can try the [experiment code for CPU-heavy task comparison](https://github.com/wkCircle/multithreading-vs-multiprocessing/blob/master/serial_comparison_cpu.py) as well, which does a simple but heavy math calculation. <br>
    Key CPU-heavy function is:
    ```python
    def cpu_heavy(x):
      print('I am', x)
      count = 0
      for i in range(10**10):
          count += i
    ```
    My output is:
    ```bash
    Serial spent 3529.652628898620
    Multithreading spent 4591.425851583481
    Multiprocessing spent 3783.1801719665527
    ```
    So indeed multithreading performs worse than serial execution on CPU-heavy task. But surprisingly, multiprocessing doesn't perform better than single thread execution. Perhaps more finer experiment should be done for a finer inspection.
  
  - it demonstrates that multithreading performs well for I/O-heavy tasks. You can try the [experiment code for I/O-heavy task comparison](https://github.com/wkCircle/multithreading-vs-multiprocessing/blob/master/serial_comparison_io.py) as well, which reads many urls. <br>
    Key I/O-heavy function is:
    ```python
    urls = [...]  # a list of many urls
    def load_url(x):
      # print('I am', x)
      with urllib.request.urlopen(urls[x], timeout=20) as conn:
          return conn.read()
    
    for i in urls:
      load_url(i)
    ```
    My output is:
    ```bash
    Serial spent 3.4708025455474854
    Multithreading 4 spent 1.1974174976348877
    Multithreading 8 spent 0.7969970703125
    Multithreading 16 spent 0.6970319747924805
    ```
    This again comfirms that multithreading is faster than single thread sequential execution for I/O-bound tasks.
    And as long as we are using more n_jobs = 4, 8, 16 in multithreading, the requested time decreases (but decreased amout are getting smaller). 
  
  - it demonstrates that multiprocessing is not neccesary always bad for I/O-bound tasks. By using the same experiment code as the previous one ([experiment code for I/O-heavy task comparison](https://github.com/wkCircle/multithreading-vs-multiprocessing/blob/master/serial_comparison_io.py)), I got the following result:
    ```bash
    Serial spent 3.4771108627319336
    Multiprocessing 4 spent 1.3483104705810547
    Multiprocessing 8 spent 1.00370454788208
    Multiprocessing 16 spent 1.0825281143188477
    ```
    This shows that multiprocessing is also very fast compared to the single thread execution, but in my case it performs a bit worse than multithreading. Hence, concluded that multithreading performs best on I/O-bound tasks, and multiprocessing on the second place.

## <a id='toc1_3_'></a>[GIL becomes optional starting from python3.13+](#toc0_)

A very interesting fact is that, starting from Python 3.13+ the official releases an experimental build with the so called Free-threaded CPython (to differentiate from standard CPython) that removes GIL. For more details, see [PEP703](https://peps.python.org/pep-0703/).
- If you want to install py3.13 without GIL, then you can go to the [official website](https://www.python.org/downloads/) and download any python 3.13 version. When installing, make sure you check the box "download free-threaded binaries (experimental)". Once installed, you will find the file name is also different than usual, called `python3.13t`, in the dedicated installation folder.
- For more details, you can follow the [YouTube video- How to Disable GIL in Python3.13](https://youtu.be/9uQoDGRs3H4) by 2MinutesPy (really just 2 minutes).
- **Warning**: according to another [YouTube video- R.I.P GIL in Python 3.13 | Will Python Be Faster Now?](https://youtu.be/mvxR6_G4yLQ) by 2MinutesPy, multithreading is indeed much faster in their demo when using `python3.13t`, however, major popular libraries such as Django or FastAPI might be broken since they use `asyncio` and `anyio` respectively, which all use `treading` package where GIL plays a major role. My comment is that we can see the trend of gradually removing the GIL in future python standard build, but it will take time for existing popular libraries to adapt to this as well.

## <a id='toc1_4_'></a>[Speed with vs without GIL](#toc0_)

In the following I also reproduce the result and confirm that No GIL mode is indeed faster than GIL mode under multithreading.

Here is my computer information:
<pre>
Processor: 12th Gen Intel(R) Core(TM) i7-12700F (2.10 GHz)
Installed RAM: 16.0 GB (15.8 GB usable)
System type: 64-bit operating system, x64-based processor

Edition: Windows 11 Home
Version: 24H2
</pre>

I have downloaded the python 3.14t version (free-threaded build), so that later I can turn on and off the GIL by simply giving the evnrionment variable `PYTHON_GIL`  or the command-line option `-X gil` (Reference: [Python Docs- The global interpreter lock in free-threaded Python](https://docs.python.org/3/howto/free-threading-python.html#the-global-interpreter-lock-in-free-threaded-python))

We will compare if python, with and without GIL lock, runs faster or not when executing single-thread/multi-thread/multi-processing on the CPU-bound tasks `compute_factorial`:
```python
import math
# target CPU-heavy task
def compute_factorial(n):
    return math.factorial(n)

# three different ways to execute the CPU-heavy task
def single_threaded_compute(n):
    for num in n:
        compute_factorial(num)
    print("Single-threaded Factorial Computed.")

def multi_threaded_compute(n):
    threads = []
    # create 5 threads
    for num in n:
        thread = threading.Thread(target=compute_factorial, args=(num,))
        threads.append(thread)
        thread.start()
    
    # wait for all threads to complete
    for thread in threads:
        thread.join()
    print("Multi-threaded Factorial Computed.")

def multi_processing_compute(n):
    processes = []
    # create a process for each number in the list
    for num in n:
        process = multiprocessing.Process(target=compute_factorial, args=(num,))
        processes.append(process)
        process.start()
    
    # wait for all processes to complete
    for process in processes:
        process.join()
    print("Multi-processing Factorial Computed.")
```

We put them under one file `gil.py` ([source code here](https://github.com/wkCircle/multithreading-vs-multiprocessing/blob/master/gil.py)) so we can execute the file via python 
either with or without GIL setting, e.g.:
```bash
# execute the file with GIL
PYTHON_GIL=1 python3.14t gil.py
# execute the file without GIL
PYTHON_GIL=0 python3.14t gil.py
```


------------- Run in GIL Mode ------------- <br>

Command:
```bash
PYTHON_GIL=1 python3.14t gil.py
```
Output:
<pre>
sys info:
3.14.0rc3 free-threading build (tags/v3.14.0rc3:1c5b284, Sep 18 2025, 09:22:50) [MSC v.1944 64 bit (AMD64)]
sys.version_info(major=3, minor=14, micro=0, releaselevel='candidate', serial=3)
Running in GIL mode 🔒
Single-threaded Factorial Computed.
Single-threaded time taken : 3.47 seconds
Multi-threaded Factorial Computed.
Multi-threaded time taken : 3.57 seconds
Multi-processing Factorial Computed.
Multi-process time taken : 1.54 seconds
</pre>


------------- Run in NO-GIL Mode ------------- <br>

Command:
```bash
PYTHON_GIL=0 python3.14t gil.py
```
Output:
<pre>
sys info:
3.14.0rc3 free-threading build (tags/v3.14.0rc3:1c5b284, Sep 18 2025, 09:22:50) [MSC v.1944 64 bit (AMD64)]
sys.version_info(major=3, minor=14, micro=0, releaselevel='candidate', serial=3)
Running in NO-GIL mode 🧵
Single-threaded Factorial Computed.
Single-threaded time taken : 3.43 seconds
Multi-threaded Factorial Computed.
Multi-threaded time taken : 1.51 seconds
Multi-processing Factorial Computed.
Multi-process time taken : 1.56 seconds
</pre>
-------------


Conclusion:

From the above experiment we see that
- multiprocessing is not affected by GIL, which makes sense since each process is run in parallel under each different core (as mentioned before).
- **multithreading without GIL is 2x faster** than multithreading with GIL mode (and is roughly the same as the Multi-processing).

## <a id='toc1_5_'></a>[Summary](#toc0_)

In this article we have introduced:
- terminologies such as core, thread, process, CPU-bound tasks, I/O-bound tasks.
- speed comparison between single-thread, multi-thread, multi-processing (under normal GIL mode using python3.12)
- understand GIL and the newly relased python3.14 and python3.14t (free threaded) version. Released on Oct 7th, 2025.
- speed comparison between python3.14t with and without GIL mode for single-thread/multi-thread/multi-processing.

We have used some key functions to setup the comparison experiments:
<table>
  <thead>
    <tr>
      <th>Function</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>compute_factorial(n)</td>
      <td>Computes factorial of <code>n</code> using <code>math.factorial()</code>. CPU-heavy.</td>
    </tr>
    <tr>
      <td>load_urls(i)</td>
      <td>Load i-th URL. I/O-heavy when giving multiple URLs.</td>
    </tr>
    <tr>
      <td>single_threaded_compute(n)</td>
      <td>Computes all factorials sequentially.</td>
    </tr>
    <tr>
      <td>multi_threaded_compute(n)</td>
      <td>Spawns one thread per computation. Shows GIL bottleneck.</td>
    </tr>
    <tr>
      <td>multi_processing_compute(n)</td>
      <td>Spawns one process per computation. Fully parallel since processes bypass GIL.</td>
    </tr>
  </tbody>
</table>

<table>
  <thead>
    <tr>
      <th>Mode</th>
      <th>Parallelism</th>
      <th>Expected Behavior</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Single-threaded</td>
      <td>None</td>
      <td>Baseline performance.</td>
    </tr>
    <tr>
      <td>Multi-threaded</td>
      <td>Limited (due to GIL, and actually is concurrency)</td>
      <td>Often same or slower than single-threaded for CPU tasks.</td>
    </tr>
    <tr>
      <td>Multi-processing</td>
      <td>True parallel</td>
      <td>Best performance on multi-core CPUs.</td>
    </tr>
  </tbody>
</table>


Performance summary for three-threaded python 3.14t:
<table>
  <thead>
    <tr>
      <th>Mode</th>
      <th>Expected Behavior</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Single-threaded</td>
      <td>Not affected by GIL setting.</td>
    </tr>
    <tr>
      <td>Multi-threaded</td>
      <td>Faster speed for No-GIL setting compared to with GIL.</td>
    </tr>
    <tr>
      <td>Multi-processing</td>
      <td>Not affected by GIL setting.</td>
    </tr>
  </tbody>
</table>


## <a id='toc1_6_'></a>[References](#toc0_)

-  [Multithreading VS Multiprocessing in Python](https://medium.com/contentsquare-engineering-blog/multithreading-vs-multiprocessing-in-python-ece023ad55a) by Amine Baatout
-  [Code base forked from Amine Baatout and adjusted](https://github.com/wkCircle/multithreading-vs-multiprocessing/tree/master)
-  [Intro to Thraeds and Processes in Python](https://medium.com/@bfortuner/python-multithreading-vs-multiprocessing-73072ce5600b) by Brendan Fortuner
-  [Difference Between Concurrency and Parallelism](https://techdifferences.com/difference-between-concurrency-and-parallelism.html) by TechDifferences
- [Python Docs- The global interpreter lock in free-threaded Python](https://docs.python.org/3/howto/free-threading-python.html#the-global-interpreter-lock-in-free-threaded-python)
- [YouTube video- How to Disable GIL in Python3.13](https://youtu.be/9uQoDGRs3H4) by 2MinutesPy
- [YouTube video- R.I.P GIL in Python 3.13 | Will Python Be Faster Now?](https://youtu.be/mvxR6_G4yLQ) by 2MinutesPy
